# 1. Relative Frequency, Chebyshev's inequality and Hoeffding bound

---
## How to work in this lab: Predict → Run → Explain → Experiment

For the highlighted investigations, **do not run the relevant code immediately**. First make a prediction and write it on your Lab Thinking Sheet. Then run the code, observe the result, explain it using concepts from the course, and finally change something to test your understanding.

**Running code is not the task. Understanding what happens when you run it is the task.**

You may use documentation or AI to help you understand or modify code. You are responsible for the prediction, the interpretation, and the explanation.
---


We first need to import the libraries that we are going to use in this notebook.

1. The *numpy* Python library is used for mathematical functions, such as our random number generators;
2. The *matplotlib* Python library is used to plot graphics and visualize data;
3. The *ipywidgets* library, for animations and widgets. 
   

In [ ]:
# NOTE: if you are in labs at DEI, open this notebook with jupyter-notebook (NOT jupyter-lab), due to outdated versions.

In [ ]:
# Imports widgets for JupyterLite
# %pip install -q ipywidgets==8.0.4 if jupyterlab with jupyterlab==4.4.9
# %pip install -q ipywidgets==7.7.1 if vscode with jupyterlab whathever

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, widgets, fixed
from IPython.display import display

random_seed = 1
np.random.seed(random_seed)

Let's define our Bernoulli variable $X_j$ simulating the flip of the $j$-th coin:

$$
X_j =
\begin{cases}
1 & \text{if H with probability $p$} \\
0 & \text{if T}
\end{cases}
$$

We know that in a Bernoulli random variable we have that:


*   $\mathbb{E} \left[ X_j \right] = p$, and
*   $\mathbb{Var} \left[ X_j \right] = p (1 - p)$.







In [ ]:
# -- function that simulates a coin flipping (Bernoulli variable)
def coin_flip(p):

  if np.random.rand() < p:
    # HEAD
    return 1
  else:
    # TAIL
    return 0

## Relative Frequency

We have a coin and we want to estimate the probability $p$ of getting HEAD, based on data observations.

Now, let's try to estimate the probability $p$ by computing the <u>relative frequency</u> simulating the flip of our coin for a number of times $i$, with $i = 1, ..., n$ and $n = 10000$.

We want to compute and plot the estimated probability for each of such flip:

\begin{equation}
  \tilde{p_i} = \sum_{j=1}^{i} \frac{X_j}{i}
\end{equation}

for $i = 1, ..., n$

Note that $\tilde{p_i}$ is the sample mean computed observing samples up to the $i$-th flip.

In [ ]:
# -- let's set our parameters
n_flips = 10000
p = np.random.rand()

In [ ]:
def estimate_probability(n_flips, p):
    estimated_p = np.zeros(n_flips)
    head_count = 0
    
    for i in range(1, n_flips+1):
        head_count += coin_flip(p)
        # -- relative frequency
        estimated_p[i - 1] = head_count/i
    
    return estimated_p

### Investigation 1 — Relative frequency

**PREDICT — Stop before running the plot.** Sketch or describe how you expect $\tilde p_i$ to behave as the number of flips increases. Will the fluctuations have the same size at the beginning and at the end? Why? Write your prediction on the Lab Thinking Sheet.

**RUN & OBSERVE.** Run the plot for different numbers of flips. Identify one feature that agrees with your prediction and, if possible, one that surprised you.

**EXPLAIN.** Explain the observed behaviour in terms of sampling variability and the Law of Large Numbers.

**EXPERIMENT.** Change $p$, the number of flips, and/or the random seed. Before each change, predict what will change and what should remain qualitatively the same.


Now, let's plot the estimated probability for each flip

In [ ]:
def plot_estimated_probability(n_flips, p):
    
    # we seed the code in order to obtain the same sequence of flips
    np.random.seed(random_seed)
    estimated_p = estimate_probability(n_flips, p)

    plt.plot(estimated_p, color = 'b', label='Estimated probability')
    
    # plot true value
    plt.axhline(y=p, color='r', linestyle='--', label='True probability')
    plt.ylim(0, 1)
    
    plt.xlabel('Number of flips')
    x_range = range(0, n_flips, max(1, int(n_flips/5)))
    plt.xticks(x_range, labels = [1 + x for x in x_range])
    plt.ylabel('Estimated probability')
    plt.legend()

In [ ]:
n_flips_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=n_flips,
    description='n_flips',
    step=1
)

In [ ]:
interact(plot_estimated_probability, n_flips = n_flips_slider, p = fixed(p));

## Not interactive relative frequency plot at increasing number of coin flips
Set the value of n_flips_not_interactive to see the behaviour of the relative frequency at increasing number of coin flips.

In [ ]:
fig = plt.Figure()
n_flips_not_interactive = 100 # set value of number of coin flips
plot_estimated_probability(n_flips_not_interactive, p)
plt.show()

## Law of Large Numbers

From the plots above, we can also see the validiy of the Law of Large Numbers, that states that given $X_i$, $i = 1, ..., n$ be i.i.d. with expectation $\mu$:

\begin{equation}
  \lim_{n\to\infty} \mathbb{P} \left[  \left| \frac{1}{n} \sum_{i = 1}^n X_i - \mu \right| > \varepsilon \right] = 0.
\end{equation}

---

In the next part, we want to compute the mean and the variance of the random variable $\tilde{p}_n$, the estimated probability for a fixed number $n$ of coin flips.IMPORTANT: $\tilde{p}_n$ can be seen as a random variable, that we recall is defined as:

\begin{equation}
  \tilde{p}_n = \sum_{j = 1}^{n} \frac{X_j}{n}
\end{equation}

So, we have that the expected value at the end of our $n$ flips is:

\begin{align}
  \mathbb{E}\left[\tilde{p}_n\right] &= \mathbb{E} \left[ \sum_{j = 1}^{n} \frac{X_j}{n} \right] = \frac{1}{n} \ \mathbb{E} \left[ \sum_{j = 1}^{n} X_j \right] = \\
  & = \frac{1}{n} \ \sum_{j = 1}^{n} \ \mathbb{E} \left[ X_j \right] = \frac{1}{n} \ \sum_{j = 1}^{n} \ p = p
\end{align}

and the variance is:

\begin{align}
  \mathbb{Var} \left[\tilde{p}_n\right] &= \mathbb{Var} \left[ \sum_{j = 1}^{n} \frac{X_j}{n} \right] = \frac{1}{n^2} \ \mathbb{Var} \left[ \sum_{j = 1}^{n} X_j \right] = \\
  & = \frac{1}{n^2} \ \sum_{j = 1}^{n} \ \mathbb{Var} \left[ X_j \right] = \frac{1}{n^2} \ n \ \mathbb{Var} \left[ X_j \right] = \frac{p (1 - p)}{n}
\end{align}

In [ ]:
# let's see if the true value of the expectation and variance of p_tilde matches the estimated one
# -- true mean
print('True mean = ', p)

# -- true variance
print('True variance = ', p*(1-p)/n_flips)

#IMPORTANT
#To make an estimate of the expectation and variance of p_tilde we need to repeat the experiments many times, 
#say m times
#Write a function that estimates the expectation and variance of p_tilde
def m_times_estimated_p(p, n_flips,m):
    p_tilde = np.random.binomial(n_flips, p, size=m) / n_flips
    mean_val = p_tilde.mean()
    var_val = p_tilde.var(ddof=1) # Bessel's correction
    return (mean_val, var_val)

p_tilde_mean, p_tilde_Var = m_times_estimated_p(p, n_flips=n_flips, m=100)

print('Estimated mean = ', p_tilde_mean)
print('Estimated Var = ', p_tilde_Var)


In [ ]:
# Lets try to do the same for diffent values of m 
# define sample sizes and initialize observed expectations and variances
sample_sizes = [10,50,100,200,400,800, 1200, 2400, 3200,6400]
X_means = []
X_vars = []

for m in sample_sizes:  # for m:= number of samples
    print("Sampling {} occurrences of a binomial random variable...".format(m))
    mean_val, var_val = m_times_estimated_p(p, n_flips=n_flips, m=m)
    X_means.append(mean_val)
    X_vars.append(var_val)
print("Done!")

### Investigation 2 — Estimating an estimator

**PREDICT.** We know theoretically that $E[\tilde p_n]=p$ and $Var(\tilde p_n)=p(1-p)/n$. If we repeat the experiment only a few times, how close do you expect the empirical mean and variance to be to these values? What should happen as the number of repetitions $m$ increases?

**RUN & OBSERVE.** Run the experiment for the proposed values of $m$. Compare empirical and theoretical quantities.

**EXPLAIN.** Why do the estimates themselves fluctuate? Why does increasing $m$ help?

**EXPERIMENT.** Change $n$, $m$, or $p$. Choose one change that you think will have a visible effect on the variance. Predict first, then test it.


Let's see if the above is true by computing the empirical mean and variance.

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(10,8))

# Left subplot: Mean
axs[0].plot(sample_sizes, X_means, "o--", label="Empirical mean")
axs[0].axhline(p, color="red", linestyle="--", label="Theoretical mean")
axs[0].set_xlabel("Sample size m")
axs[0].set_ylabel("Mean of estimator")
axs[0].set_title(r"Convergence of $E[\tilde{{p}}_n]$ to p with n flips = {}".format(n_flips))
axs[0].set_ylim([min(min(X_means), 0.99*p), max(max(X_means), 1.01*p)])
axs[0].legend()

# Right subplot: Variance
axs[1].axhline(p*(1-p)/n_flips, color="red", linestyle="--", label = "Theoretical variance")
axs[1].plot(sample_sizes, X_vars, "o--", label = "Empirical variance")
axs[1].set_xlabel("Sample size m")
axs[1].set_ylabel("Variance of estimator")
axs[1].set_title(r"Convergence of $\mathrm{{Var}}[\tilde{{p}}_n]$ to $\frac{{p(1-p)}}{{n}}$ with n flips = {}".format(n_flips))

axs[1].legend()

plt.tight_layout()
plt.show()

## Chebyshev's inequality

As you have seen during the lectures, given a random variable $X$ with $\mathbb{E}\left[X\right] = \mu$ and $\mathbb{Var}\left[X\right] = \sigma^2$, and a fixed constant $\varepsilon > 0 $, the Chebyshev's inequality states:

\begin{equation}
  \mathbb{P}\left[ \left| X - \mu \right| > \varepsilon \right] \leq \frac{\sigma^2}{\varepsilon^2}
\end{equation}

In our example, with Binomial random variables, we want to plot the Chebyshev bounds given by the above inequality for the estimated probability in each of our coin flips.

So, if we let:

  * $X = \tilde{p_i}$
  * $\mu = \mathbb{E}\left[ \tilde{p_i} \right] = p$
  * $\sigma_i^2 = \mathbb{Var}\left[ \tilde{p_i} \right] = \frac{p (1 - p)}{i}$
  * $\varepsilon = \varepsilon$

we obtain:

\begin{equation}
  \mathbb{P}\left[ | \tilde{p_i} - p | > \varepsilon  \right] \leq \frac{p (1 - p)}{i \varepsilon^2 }
\end{equation}



Let's compute and plot the probability bounds of being far from the true mean, for each of our $\tilde{p_i}$ random variables, $i = 1, ..., n$.

Let's fix the right-hand-side of the above inequality, by letting $\delta = \frac{p (1 - p)}{i  \epsilon^2} = 0.05$, that corresponds to the probability of obtaining "bad" estimates. We need to compute the distance from the mean (i.e., $\varepsilon$) for each number of n_flips $i$, that is:

\begin{equation}
  \varepsilon  = \sqrt{\frac{p(1 - p)}{\delta i}}
\end{equation}

In [ ]:
estimated_p = estimate_probability(n_flips,p)

def plot_chebyshev_bounds(delta, estimated_p, n_flips, p):

    flips = np.array([i for i in range(1, n_flips+1)])
    
    
    plt.plot(estimated_p, color = 'b', label='Estimated probability')
    # -- plot true value
    plt.axhline(y = p, color = 'r', linestyle = '--', label='True probability')
    
    # -- Chebyshev's bounds: we compute the epsilon for each flip
    chebyshev_values = np.sqrt((p * (1 - p)) / (flips * delta))
    plt.plot(p + chebyshev_values, color = 'g', linestyle='--')
    plt.plot(p - chebyshev_values, color = 'g', linestyle='--', label=f'Chebyshev bounds with delta={round(delta, 3)}')
    plt.ylim(0, 1)
    
    
    plt.xlabel('Number of flips')
    plt.ylabel('Estimated probability')
    plt.legend()

Try with different values of $\delta$, that reflect the probability of obtaining bad approximation.

In [ ]:
delta_slider = widgets.FloatSlider(
    value=0.005,
    min=0.001,
    max=1.0,
    description='delta',
    step=0.005
)

In [ ]:
interact(plot_chebyshev_bounds, delta = delta_slider, estimated_p = fixed(estimated_p), n_flips = fixed(n_flips), p = fixed(p));

## Not interactive Chebyshev's Inequality bounds plot
Set the value of delta and re run the cell to see the behaviour of the chebyshev's inequality bounds at varying of delta parameter.

In [ ]:
fig = plt.Figure()
delta = 0.05 # set value of delta to draw chebyshev bounds
plot_chebyshev_bounds(delta, estimated_p, n_flips, p)
plt.show()

## Hoeffding's bound

Now, we are going to analyze a more precise tool for the approximation guarantees: the Hoeffding's bound (you will in the lectures as well).




**Hoeffding's bound:**
Given the sample average  $\bar X = \frac{1}{i} \sum_{j=1}^i X_j$ of Bernoulli random variables $X_i$ with $\mu = \mathbb{E} \left[\bar X \right]$, and a small constant $\varepsilon \in (0, 1)$, we have that:

\begin{equation}
  \mathbb{P}\left[ \left| \bar{X} - \mu \right| > \varepsilon \right] \leq 2 e^{-2i \varepsilon^2 }
\end{equation}

In particular, as done for the Chebyshev's bounds) note that $\tilde{p_i}$ is the sample average over $i$ trials of Bernoulli trials (the coin flips) with mean $\mu$

\begin{align}
  & \mathbb{P}\left[ | \tilde{p_i} - p | > \varepsilon \right] \\
  & = \mathbb{P}\left[ \big | \frac{1}{i}\sum_{j=1}^i X_j - p \ \big | > \varepsilon \right] \\
  & \leq 2 e^{-2 i \varepsilon^2 }
\end{align}

Also in this case, let's fix $\delta = 2 e^{ - 2 i \varepsilon^2 } = 0.05$, the probability of optaining "bad" estimates.



We can similarly plot the Hoeffding's bounds by computing the distances from the true mean (i.e., $\varepsilon$ as defined in the above) for each coin flip.

In [ ]:
def plot_hoeffding_bound(delta, estimated_p, n_flips, p):

    flips = np.array([i for i in range(1, n_flips+1)])
    plt.plot(estimated_p, color = 'b', label='Estimated probability')
    # plot true value
    plt.axhline(y = p, color = 'r', linestyle = '--', label='True probability')
    
    plt.ylim(0, 1)
    
    # -- Hoeffding's bounds: let's compute the p * epsilon values
    #hoeffding_values = np.sqrt(((np.log(2/delta)) * 3 * p) / flips)
    hoeffding_values = np.sqrt( ( np.log(2/delta) ) / (2*flips))
    plt.plot(p + hoeffding_values, color = 'k', linestyle='--')
    plt.plot(p - hoeffding_values, color = 'k', linestyle='--', label=f'Hoeffding bounds with delta={round(delta, 3)}')
    
    
    plt.xlabel('Number of flips')
    plt.ylabel('Estimated probability')
    plt.legend()

In [ ]:
interact(plot_hoeffding_bound, delta = delta_slider, estimated_p = fixed(estimated_p), n_flips = fixed(n_flips), p = fixed(p));

## Not interactive Hoeffding's Inequality bounds plot
Set the value of delta and re run the cell to see the behaviour of the Hoeffding's inequality bounds at varying of delta parameter.

In [ ]:
fig = plt.Figure()
delta = 0.05 # set value of delta to draw hoeffding bounds
plot_hoeffding_bound(delta, estimated_p, n_flips, p)
plt.show()

### Investigation 3 — Comparing probability bounds

**PREDICT.** Before plotting both bounds, which one do you expect to be tighter in this Bernoulli setting: Chebyshev or Hoeffding? How do you expect the width of the bounds to change as the number of flips increases?

**RUN & OBSERVE.** Plot the bounds and compare them across the range of flips.

**EXPLAIN.** Explain why a valid bound does not have to closely track the observed error in one particular run. What does the bound guarantee?

**EXPERIMENT.** Vary $\delta$ and $p$. Find a setting where the visual difference between the two bounds is especially clear. Explain what you found.


Let's compare Chebyshev's and Hoeffding's bounds

In [ ]:
def plot_chebyshev_and_hoeffding_bound(delta, estimated_p, n_flips, p):

    flips = np.array([i for i in range(1, n_flips+1)])

    plt.plot(estimated_p, color = 'b', label='Estimated probability')
    # plot true value
    plt.axhline(y = p, color = 'r', linestyle = '--', label='True probability')
    
    plt.ylim(0, 1)
    
    # -- Chebyshev's bounds
    chebyshev_values = np.sqrt((p * (1 - p)) / (flips * delta))
    plt.plot(p + chebyshev_values, color = 'g', linestyle='--')
    plt.plot(p - chebyshev_values, color = 'g', linestyle='--', label=f'Chebyshev bounds with delta={round(delta, 3)}')
    
    # -- hoeffding's bounds
    hoeffding_values = np.sqrt(np.abs((np.log(2/delta)) )/(2*flips))
    plt.plot(p + hoeffding_values, color = 'k', linestyle='--')
    plt.plot(p - hoeffding_values, color = 'k', linestyle='--', label=f'Hoeffding bounds with delta={round(delta, 3)}')
    
    
    plt.xlabel('Number of flips')
    plt.ylabel('Estimated probability')
    plt.legend()

In [ ]:
interact(plot_chebyshev_and_hoeffding_bound, delta = delta_slider, estimated_p = fixed(estimated_p), n_flips = fixed(n_flips), p = fixed(p));

## Not interactive Chebyshev's and Hoeffding's Inequality bounds plot
Set the value of delta and re run the cell to see the behaviour of the Chebyshev's and Hoeffding's inequality bounds at varying of delta parameter.

In [ ]:
fig = plt.Figure()
delta = 0.05 # set value of delta to draw chebyshev and Hoeffding bounds
plot_chebyshev_and_hoeffding_bound(delta, estimated_p, n_flips, p)
plt.show()

# 2. Coin flipping: a Machine Learning case study

We will now exploit the concepts we have introduced to build a simplified Machine Learning case study starting from simple coin flippings. We now consider a coin C modeled as a Bernoulli random variable $C \sim {\mathcal B(p)}$ with $p$ the probability of head (encoded as $C=1$) and let $Y = \frac{1}{n} \sum_{i=1}^n C_i$ the average number of successes (heads) over $n=10$ (fixed) i.i.d. trials. 

Our goal is to find a model that predict the average number of heads in $n=10$ trials. Observe that in this model there are no input features, and we can only predict the label $Y$ with a constant. We consider the following class of models
$$
{\mathcal H}_k :=\{\theta \in \{\theta_1,..,\theta_{n_k}\}, \theta_i \in [0,1]\}
$$

The integer $k$ identifies the class ${\mathcal H}_k$ that is finite with cardinality $n_k$. For simplicity we assume 
${\mathcal H}_0 := \{\theta \in \{0,p,1\}\}$ with $p$ the true probability of the coin. (This is of course irrealistic in practice but it makes our life easier today.)

We consider a sequence of nested clasess
$$
{\mathcal H}_0 \subset {\mathcal H}_1 \subset {\mathcal H}_2 \subset {\mathcal H}_3 \subset {\mathcal H}_4 \subset ...
$$
with cardinalities $n_0 = 3 < n_1 < n_2 < n_3 < ..$ obtained recursively adding candiadate $\theta$'s to the previous (less complex) class. 


We assume our training set  is ${\mathcal S}:=\{ y_1,..y_m\}$ i.i.d. trials of $Y$ and consider the square loss
$$
\ell(\theta,y) = (y-\theta)^2
$$
so that the empirical risk is
$$
L_{\mathcal S}(\theta) = \frac{1}{m} \sum_{i=1}^m (y_i-\theta)^2
$$
and the **true risk** is 
$$
L_{\mathcal D}(\theta) = {\mathbb E}_{y\sim {\mathcal D}} (y-\theta)^2 = \frac{p(1-p)}{n} + (p-\theta)^2=\underbrace{\frac{p(1-p)}n}_{\sigma_Y^2}+(p-\theta)^2
$$
where the last equality follows from the fact that $y$ is obtained scaling by a factor $1/n$ Binomilal r.v. with parameters $p$ and $n$, so that its mean is $p$ and its variance if $p(1-p)/n$.

Where does the expression of $L_{\mathcal D}(\theta)$ come from?
We are using the squared loss. The true risk is

$$
L_{\mathcal D}(\theta)
=
\mathbb E_{Y\sim\mathcal D}\left[(Y-\theta)^2\right].
$$

Let's add and subtract the true mean $p$ to $Y-\theta$:

$$
Y-\theta=(Y-p)+(p-\theta).
$$

Therefore,

$$
\begin{aligned}
L_{\mathcal D}(\theta)
&=
\mathbb E\left[\big((Y-p)+(p-\theta)\big)^2\right] \\
&=
\mathbb E[(Y-p)^2]
+
2(p-\theta)\mathbb E[Y-p]
+
(p-\theta)^2.
\end{aligned}
$$

Since

$$
\mathbb E[Y-p]=0,
$$

the cross term vanishes, and

$$
L_{\mathcal D}(\theta)
=
\mathrm{Var}(Y)
+
(p-\theta)^2.
$$

Because

$$
\mathrm{Var}(Y)=\frac{p(1-p)}{n},
$$

we obtain

$$
L_{\mathcal D}(\theta)
=
\underbrace{\frac{p(1-p)}{n}}_{\sigma_Y^2}
+
(p-\theta)^2.
$$

The first term, $\sigma_Y^2$, is the variance of $Y$ and does not depend on $\theta$. The second term measures how far the prediction $\theta$ is from the true mean $p$.

Therefore, the true risk is minimized when

$$
\theta=p,
$$

and the minimum possible true risk is

$$
L_{\mathcal D}(p)=\sigma_Y^2=\frac{p(1-p)}{n}.
$$





Now we select the ERM fo each class ${\mathcal H}_k$ as follows:

$$
\hat\theta_k := \displaystyle{{\mathop{\rm arg\; min}}_{\theta \in {\mathcal H}_k}} L_{\mathcal S}(\theta).
$$

The signed generalisation gap is $L_{\mathcal D}(\hat\theta_k)-L_{\mathcal S}(\hat\theta_k)$.



In [ ]:

# Edit these parameters, then Run All.
p = 0.45            # True probability; 0 < p < 1 gives |H_0| = 3.
n = 10              # Coin flips per label (fixed in the case study).
m = 5              # Independent labels per training set.
K = 10               # Classes H_0, ..., H_K-1.
seed = abs(int(10*np.random.normal()))
repetitions = 100  # Independent training sets for the optional MC experiment.

assert 0 < p < 1
assert n == 10
assert isinstance(m, int) and m >= 1
assert isinstance(K, int) and K >= 1
assert isinstance(repetitions, int) and repetitions >= 1
sigma2 = p * (1 - p) / n
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True,
                     'grid.alpha': 0.25, 'font.size': 11})

## 2.1 Increasingly rich hypothesis classes

We now need to construct the increasingly rich hypothesis classes

$$
\mathcal H_0
\subset
\mathcal H_1
\subset
\mathcal H_2
\subset
\cdots.
$$

Remember what this means: every element of $H_k$ is simply a **candidate value for the prediction $\theta$**.

We start from

$$
\mathcal H_0=\{0,p,1\}.
$$

To make the hypothesis class richer, we progressively add more candidate values between \(0\) and \(1\).

In particular, we can progressively add new candidate values in the gaps between the values that are already available.

### How does it work?

Suppose, for example, that

$$
p=0.45.
$$

We start with

$$
\mathcal H_0=\{0,\;0.45,\;1\}.
$$

There are two gaps between consecutive candidate values:

$$
[0,0.45]
\qquad\text{and}\qquad
[0.45,1].
$$

We identify the **largest gap** and insert its midpoint. In this example, the largest gap is $[0.45,1]$, so we add

$$
\frac{0.45+1}{2}=0.725.
$$

The set of candidate predictions then becomes

$$
\{0,\;0.45,\;0.725,\;1\}.
$$


We repeat the same procedure: identify the largest remaining gap, insert its midpoint, and continue.

In the implementation below, we perform this refinement several times to construct each successive hypothesis class. Therefore,

$$
\mathcal H_0
\subset
\mathcal H_1
\subset
\mathcal H_2
\subset
\cdots,
$$

with each class containing more candidate predictions than the preceding one.

The purpose is to make the set of possible predictions progressively **finer**, so that ERM has more candidate values of $\theta$ from which to choose.

The parameters `K` and `elements_per_class` control, respectively, the number of hypothesis classes and the number of new candidate values added when moving from one class to the next.

Note that $\mathcal H_0$ contains the true value $p$. This is the deliberate **oracle assumption** of our simplified experiment.

Finally, all hypothesis classes are constructed **before the training sample is generated**. They do not depend on the particular observations in the training set.

The following code constructs these nested hypothesis classes and displays their sizes.

In [ ]:
def nested_classes(p, K, elements_per_class):
    if not 0 < p < 1:
        raise ValueError("p must be strictly between 0 and 1.")
    if K < 1 or elements_per_class < 1:
        raise ValueError("Both counts must be positive integers.")

    classes = [np.array([0.0, p, 1.0])]

    for _ in range(K - 1):
        current = classes[-1].copy()

        for _ in range(elements_per_class):
            gaps = np.diff(current)
            i = np.argmax(gaps)
            midpoint = (current[i] + current[i + 1]) / 2
            current = np.insert(current, i + 1, midpoint)

        classes.append(current)

    return classes

elements_per_class = 10 # New parameters added to each successive class

classes = nested_classes(p, K, elements_per_class)
sizes = np.array([len(h) for h in classes])

display(pd.DataFrame({
    'k': np.arange(K),
    'n_k': sizes
}))

## 2.2 Sample the training set and fit the ERM

Let us briefly recap where we are.

We have defined:

- a random variable $Y$, representing the fraction of heads observed in $n$ coin flips;
- a training set

$$
\mathcal S=\{y_1,\ldots,y_m\},
$$

consisting of $m$ independent observations of $Y$;

- the squared loss

$$
\ell(\theta,y)=(y-\theta)^2;
$$

- a sequence of nested hypothesis classes

$$
\mathcal H_0\subset\mathcal H_1\subset\mathcal H_2\subset\cdots,
$$

where each $\mathcal H_k$ contains a finite set of possible constant predictions $\theta$.

We are now ready to **train our model**.

### Step 1 — Sample the training set

We first generate $m$ independent observations

$$
y_1,\ldots,y_m
$$

from the distribution of $Y$.

These observations form the training set available to the learner.

Let

$$
\bar y=\frac{1}{m}\sum_{i=1}^{m}y_i
$$

denote their sample mean.

### Step 2 — Select the ERM in each hypothesis class

For each hypothesis class $\mathcal H_k$, we select the candidate $\theta$ that minimizes the empirical risk:

$$
\hat{\theta}_k
=
\underset{\theta\in\mathcal H_k}{\operatorname{argmin}}
\;
L_{\mathcal S}(\theta),
$$

where

$$
L_{\mathcal S}(\theta)
=
\frac{1}{m}\sum_{i=1}^{m}(y_i-\theta)^2.
$$

With the squared loss, we can rewrite the empirical risk as

$$
L_{\mathcal S}(\theta)
=
\frac{1}{m}\sum_{i=1}^{m}(y_i-\bar y)^2
+
(\bar y-\theta)^2.
$$

The first term does not depend on $\theta$. Therefore, minimizing the empirical risk is equivalent to minimizing

$$
(\bar y-\theta)^2.
$$

Hence, in each hypothesis class, the ERM has a particularly simple interpretation:

> **$\hat{\theta}_k$ is the candidate in $\mathcal H_k$ that is closest to the sample mean $\bar y$.**

The code below will generate one training set and then find the ERM $\hat{\theta}_k$ for each of our hypothesis classes.

In [ ]:
def empirical_risk(theta, y):
    theta = np.asarray(theta)
    return np.mean((np.asarray(y)[:, None] - theta.reshape(1, -1))**2, axis=0)

def true_risk(theta, p, n=10):
    return p * (1 - p) / n + (np.asarray(theta) - p)**2

# Implementation note.
#If two candidates are numerically at the same distance from $\bar y$, the code selects the smaller one. 
#This is only a tie-breaking convention and does not affect the main idea of the experiment.
def nearest_candidates(means, h):
    distances = (np.atleast_1d(means)[:, None] - h[None, :])**2
    minima = distances.min(axis=1, keepdims=True)
    tied = np.isclose(distances, minima, rtol=0, atol=1e-14)
    return h[tied.argmax(axis=1)]  # h is sorted: smallest tied candidate.

rng = np.random.default_rng(seed)
y = rng.binomial(n, p, size=m) / n
y_mean = y.mean()
print('Training labels:', y)
print(f'Sample mean = {y_mean:.6f}; true p = {p:.6f}')

rows = []
for k, h in enumerate(classes):
    theta_hat = nearest_candidates(y_mean, h)[0]
    training = empirical_risk(theta_hat, y)[0]
    population = float(true_risk(theta_hat, p, n))
    rows.append({'k': k, 'n_k': len(h), 'theta_hat': theta_hat,
                 'empirical_risk': training, 'true_risk': population,
                 'generalisation_gap': population - training,
                 'excess_risk': population - sigma2})
results = pd.DataFrame(rows)
display(results.round(6))
print(f'Optimal population risk in every class: {sigma2:.6f}')

### Reading the results

The table above summarizes the result of ERM for **one training set**.

Each row corresponds to one hypothesis class $\mathcal H_k$.

The columns report:

- **`k`** — the index of the hypothesis class;
- **`n_k`** — the number of candidate models in $\mathcal H_k$;
- **`theta_hat`** — the model $\hat\theta_k$ selected by ERM;
- **`empirical_risk`** — the risk of $\hat\theta_k$ measured on the training set,

$$
L_{\mathcal S}(\hat\theta_k);
$$

- **`true_risk`** — the expected risk of the same model on a new observation from the true distribution,

$$
L_{\mathcal D}(\hat\theta_k);
$$

- **`generalisation_gap`** — the difference between true and empirical risk,

$$
L_{\mathcal D}(\hat\theta_k)
-
L_{\mathcal S}(\hat\theta_k);
$$

- **`excess_risk`** — how much larger the true risk of the selected model is than the minimum possible true risk,

$$
L_{\mathcal D}(\hat\theta_k)-L_{\mathcal D}(p).
$$

Before looking at the plots, inspect the table.

As $k$ increases and the hypothesis class becomes richer:

1. How does $\hat\theta_k$ change?
2. What happens to the empirical risk?
3. What happens to the true risk?
4. Do empirical risk and true risk necessarily move in the same direction?

The plots below show the same information visually.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
axes[0].plot(results.k, results.theta_hat, 'o-', label='ERM')
axes[0].axhline(p, color='black', linestyle='--', label='True p')
axes[0].axhline(y_mean, color='tab:orange', linestyle=':', label='Sample mean')
axes[0].set(ylabel='Prediction', title='Fitted parameter')
axes[1].plot(results.k, results.empirical_risk, 'o-', label='Empirical risk')
axes[1].plot(results.k, results.true_risk, 's-', label='True risk')
axes[1].axhline(sigma2, color='black', linestyle='--', label='Optimal true risk')
axes[1].set(ylabel='Squared loss', title='Risk at the ERM')
axes[2].plot(results.k, results.generalisation_gap, 'o-', label='True − empirical')
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set(ylabel='Signed gap', title='One training set')
for ax in axes:
    ax.set_xlabel('Class index k')
    ax.legend(fontsize=9)
plt.show()

### Observe the results for one training set

The three plots above describe what happened for **one particular training set** $\mathcal S$.

Let us read them carefully.

**Left — Fitted parameter.**  
For each hypothesis class $\mathcal H_k$, the plot shows the parameter $\hat\theta_k$ selected by ERM. The horizontal lines show the true value $p$ and the sample mean $\bar y$.

**Center — Empirical and true risk.**  
For each selected $\hat\theta_k$, we compare:

$$
L_{\mathcal S}(\hat\theta_k),
$$

the empirical risk measured on the training data, and

$$
L_{\mathcal D}(\hat\theta_k),
$$

the true risk under the data-generating distribution.

**Right — Generalization gap.**  
The third plot shows the signed difference

$$
L_{\mathcal D}(\hat\theta_k)
-
L_{\mathcal S}(\hat\theta_k).
$$

Before continuing, discuss with your partner:

1. As the hypothesis class becomes richer, what happens to the selected $\hat\theta_k$?
2. What happens to the empirical risk?
3. Does the true risk behave in the same way?
4. What does the generalization gap tell us in this particular experiment?

Remember: all these plots refer to **one randomly generated training set**. If we generated another training set, we would generally obtain different values of $\bar y$, different ERM solutions, and different risks.

# 3. Expected generalization gap: repeated training sets

So far, we have studied what happens for **one particular training set**.

But the training set itself is random.

If we repeat the entire experiment, we obtain new observations, a different sample mean, and potentially different ERM solutions $\hat\theta_k$.

Therefore, the generalization gap observed in the previous section is also a random quantity: its value depends on the particular training set that happened to be sampled.

To understand the typical behavior of the learning procedure, we now repeat the experiment many times.

For each independently generated training set, we:

1. generate $m$ new training labels;
2. fit the ERM $\hat\theta_k$ for each hypothesis class;
3. compute its empirical risk;
4. compute its true risk;
5. compute the corresponding generalization gap.

We can then average these quantities across many independent training sets.

In particular, we are interested in the **expected generalization gap**

$$
\mathbb E_{\mathcal S}
\left[
L_{\mathcal D}(\hat\theta_k(\mathcal S))
-
L_{\mathcal S}(\hat\theta_k(\mathcal S))
\right].
$$

The expectation is now over the random choice of the **training set** $\mathcal S$.

We approximate this expectation by Monte Carlo simulation: we repeatedly generate independent training sets, fit the models again, and average the resulting generalization gaps.

**Within each repetition**, all hypothesis classes are evaluated using the **same training set**, so that their results can be compared under the same sampled data.

The true risk is still computed analytically from the known data-generating distribution; we do not need to generate a separate test set.

In [ ]:
mc_rng = np.random.default_rng(seed + 1)
training_sets = mc_rng.binomial(n, p, size=(repetitions, m)) / n
means = training_sets.mean(axis=1)
within_variances = training_sets.var(axis=1, ddof=0)
mc_rows = []
for k, h in enumerate(classes):
    estimates = nearest_candidates(means, h)
    train = within_variances + (means - estimates)**2
    true = true_risk(estimates, p, n)
    gaps = true - train
    mc_rows.append({'k': k, 'n_k': len(h),
                    'expected_empirical_risk_MC': train.mean(),
                    'expected_true_risk_MC': true.mean(),
                    'expected_gap_MC': gaps.mean(),
                    'gap_MC_standard_error': gaps.std(ddof=1) / np.sqrt(repetitions)})
mc = pd.DataFrame(mc_rows)
display(mc.round(6))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(mc.k, mc.expected_empirical_risk_MC, 'o-', label='Mean empirical risk')
axes[0].plot(mc.k, mc.expected_true_risk_MC, 's-', label='Mean true risk')
#axes[0].axhline(sigma2, color='black', linestyle='--', label='Optimal true risk')
axes[0].set(xlabel='Class index k', ylabel='Squared loss', title='Averages over training sets')
axes[1].errorbar(mc.k, mc.expected_gap_MC,
                 yerr=1.96 * mc.gap_MC_standard_error, fmt='o-', capsize=4,
                 label='Mean gap ± 1.96 MC standard errors')
axes[1].axhline(2 * sigma2 / m, color='tab:orange', linestyle='--',
                label='Continuous-class expected gap')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set(xlabel='Class index k', ylabel='Signed gap', title='Expected generalisation gap (MC)')
for ax in axes:
    ax.legend(fontsize=9)
plt.show()

# 4. Interpreting the results

We can now use the experiments above to understand what happens when the hypothesis class becomes richer.

Let us consider the main observations one at a time.

## 4.1 What happens to the empirical risk?

Recall that the hypothesis classes are nested:

$$
\mathcal H_0
\subset
\mathcal H_1
\subset
\mathcal H_2
\subset
\cdots.
$$

Therefore, every model that is available in $\mathcal H_k$ is still available in $\mathcal H_{k+1}$, together with some additional candidates.

ERM chooses the model with the smallest empirical risk.

Hence, enlarging the hypothesis class cannot make the minimum empirical risk worse:

$$
\min_{\theta\in\mathcal H_{k+1}}
L_{\mathcal S}(\theta)
\leq
\min_{\theta\in\mathcal H_k}
L_{\mathcal S}(\theta).
$$

In our example, a richer hypothesis class gives ERM more candidate values and therefore allows it to get closer to the observed sample mean $\bar y$.

> **A richer hypothesis class can fit the training data at least as well as a smaller one.**

This explains why the empirical risk is non-increasing as the hypothesis class becomes richer.

---

## 4.2 Does a smaller empirical risk imply a smaller true risk?

Not necessarily.

The empirical risk is computed using the particular training sample that we observed.

As seen before, the true risk is

$$
L_{\mathcal D}(\theta)
=
\sigma_Y^2+(p-\theta)^2,
$$

where

$$
\sigma_Y^2=\frac{p(1-p)}{n}.
$$

Therefore, the true risk is smallest when $\theta=p$.

But ERM does not directly try to get close to $p$: it minimizes the empirical risk and therefore tries to get close to the **sample mean** $\bar y$.

For a finite training set,

$$
\bar y \neq p
$$

in general.

Thus, making the hypothesis class richer may allow ERM to fit the particular fluctuations of the training sample more closely.

This can reduce the empirical risk without necessarily reducing the true risk.

> **Better fit to the training sample does not automatically mean better performance on the underlying distribution (i.e., in terms of true risk).**

---

## 4.3 Approximation error and excess risk

There is a special feature of our simplified experiment.

By construction,

$$
p\in\mathcal H_k
$$

for every hypothesis class.

Therefore, every class contains the model that minimizes the true risk.

The best possible true risk within every $\mathcal H_k$ is consequently

$$
L_{\mathcal D}(p)
=
\sigma_Y^2.
$$

So, in this particular experiment, making the hypothesis class richer does **not** improve the best model that is available: the optimal model $p$ was already present in $\mathcal H_0$.


We can now define the **excess risk** of the model selected by ERM:

$$
L_{\mathcal D}(\hat\theta_k)
-
L_{\mathcal D}(p).
$$

Using

$$
L_{\mathcal D}(\theta)
=
\sigma_Y^2+(p-\theta)^2,
$$

we obtain

$$
L_{\mathcal D}(\hat\theta_k)
-
L_{\mathcal D}(p)
=
(\hat\theta_k-p)^2.
$$

Thus, in this example, the excess risk simply measures how far the model selected from the training data is from the optimal prediction $p$.

---

## 4.4 What does the generalization gap tell us?

Recall that the signed generalization gap is

$$
L_{\mathcal D}(\hat\theta_k)
-
L_{\mathcal S}(\hat\theta_k).
$$

It compares the performance of the **same learned model** on the training sample with its expected performance on the underlying distribution.

For one particular training set, this gap may be positive or negative.

A negative value is not a contradiction: empirical risk and true risk are different quantities, and the particular training sample is random.

This is why, in the previous section, we repeated the experiment over many independent training sets and studied the **expected generalization gap** rather than drawing conclusions from a single realization.



## 5 A useful limiting case: the continuous hypothesis class

Our finite hypothesis classes contain increasingly many candidate values of $\theta$.

We can ask what would happen in the limiting case in which the learner could choose **any**

$$
\theta\in[0,1].
$$

For the continuous hypothesis class

$$
\mathcal H=[0,1],
$$

the ERM is simply

$$
\hat\theta=\bar Y,
$$

because the sample mean itself is now an available candidate.

For this continuous class, the expected empirical and true risks can be computed analytically:

$$
\mathbb E_{\mathcal S}
\left[
L_{\mathcal S}(\bar Y)
\right]
=
\sigma_Y^2
\left(
1-\frac{1}{m}
\right),
$$

and

$$
\mathbb E_{\mathcal S}
\left[
L_{\mathcal D}(\bar Y)
\right]
=
\sigma_Y^2
\left(
1+\frac{1}{m}
\right).
$$

Therefore,

$$
\mathbb E_{\mathcal S}
\left[
L_{\mathcal D}(\bar Y)
-
L_{\mathcal S}(\bar Y)
\right]
=
\frac{2\sigma_Y^2}{m}.
$$

This gives us an **analytic benchmark** against which we can compare the increasingly fine finite hypothesis classes.

As the grids become finer, their behavior can approach that of the continuous class, although a finite hypothesis class does not have to give exactly the same result.

In [ ]:
continuous_train = within_variances
continuous_true = true_risk(means, p, n)
benchmark = pd.DataFrame({
    'quantity': ['Expected empirical risk', 'Expected true risk', 'Expected gap'],
    'analytic': [sigma2 * (1 - 1/m), sigma2 * (1 + 1/m), 2 * sigma2 / m],
    'Monte Carlo': [continuous_train.mean(), continuous_true.mean(),
                    (continuous_true - continuous_train).mean()]})
display(benchmark.round(6))

# Mathematical consistency checks.
assert all(p in h for h in classes)
assert all(set(a) < set(b) for a, b in zip(classes, classes[1:]))
assert np.all(np.diff(results.empirical_risk) <= 1e-12)
assert np.allclose(results.excess_risk, (results.theta_hat - p)**2)
# Independent calculation: sum the loss over the exact Binomial label distribution.
from math import comb
support = np.arange(n + 1) / n
probabilities = np.array([comb(n, j) * p**j * (1-p)**(n-j) for j in range(n+1)])
for row, h in zip(results.itertuples(), classes):
    assert np.isclose(row.empirical_risk, empirical_risk(h, y).min())
    exact_sum = np.sum(probabilities * (support - row.theta_hat)**2)
    assert np.isclose(row.true_risk, exact_sum)
print('All consistency checks passed.')

---
# End-of-lab discussion — Close the notebook

Before the class discussion, take **2 minutes individually** and answer from memory:

1. What is the most important connection you saw today between probability and learning from data?
2. Why can excellent training performance be misleading?
3. Name one quantity you changed today and explain what effect it had and why.
4. What is one result from the lab that you could now explain without looking at the notebook?
5. What is still unclear? Write one specific question.

Then compare your answers with your partner. Be ready to share **one claim and the evidence from an experiment that supports it**.
